# Lọc 5 core file df_rating_2023.parquet để tạo ra file df_inter.parquet có 2 cột userID và itemID đã được map sang số nguyên để dễ xử lý ở bước sau.

## Bước này tạo ra file:

- df_inter.parquet: id của người và sản phẩm đã chuyển sang số vẫn lưu 2 cột để map

Vì code cũ tác giả có đánh label nhưng qua bước 1 thì chuyển xài label khác nên bước này chỉ map id thôi.


# 5-core filtering

- Extracting U-I interactions and performing 5-core, re-indexing
- dataset located at: http://jmcauley.ucsd.edu/data/amazon/links.html, rating only file in "Small" subsets for experimentation


In [1]:
import os
import pandas as pd

In [2]:
PATH = "../data/2023"

In [4]:
os.makedirs(os.path.join(PATH, "step1_rating_to_inter"), exist_ok=True)

In [ ]:
df = pd.read_parquet(os.path.join(PATH, "step0_clean_data", "df_rating.parquet"))

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6025848 entries, 0 to 6025847
Data columns (total 4 columns):
 #   Column     Dtype
---  ------     -----
 0   userID     str  
 1   itemID     str  
 2   rating     int64
 3   timestamp  int64
dtypes: int64(2), str(2)
memory usage: 402.3 MB


In [7]:
df.shape

(6025848, 4)

In [8]:
df.head(3)

,userID,itemID,rating,timestamp
0,AGKASBHYZPGTEPO6LWZPVJWB2BVA,B089MS68G8,4,1471546337000
1,AGKASBHYZPGTEPO6LWZPVJWB2BVA,B01E5E703G,5,1471542244000
2,AGKASBHYZPGTEPO6LWZPVJWB2BVA,B00F9386Q8,1,1452650881000


## 5-core filtering


In [9]:
print(f"shape: {df.shape}")
df[:5]

shape: (6025848, 4)


,userID,itemID,rating,timestamp
0,AGKASBHYZPGTEPO6LWZPVJWB2BVA,B089MS68G8,4,1471546337000
1,AGKASBHYZPGTEPO6LWZPVJWB2BVA,B01E5E703G,5,1471542244000
2,AGKASBHYZPGTEPO6LWZPVJWB2BVA,B00F9386Q8,1,1452650881000
3,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B07RRDX26B,5,1408994051000
4,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B00OLRJET6,5,1349818961000


In [10]:
# Bỏ giá trị null với trùng
learner_id, course_id = "userID", "itemID"

df.dropna(subset=[learner_id, course_id], inplace=True)
df.drop_duplicates(subset=[learner_id, course_id], inplace=True)
print(f"After dropped: {df.shape}")
df[:3]

After dropped: (5950878, 4)


,userID,itemID,rating,timestamp
0,AGKASBHYZPGTEPO6LWZPVJWB2BVA,B089MS68G8,4,1471546337000
1,AGKASBHYZPGTEPO6LWZPVJWB2BVA,B01E5E703G,5,1471542244000
2,AGKASBHYZPGTEPO6LWZPVJWB2BVA,B00F9386Q8,1,1452650881000


In [11]:
from collections import Counter
import numpy as np

min_u_num, min_i_num = 5, 5


# Hàm này có nhiệm vụ tìm ra danh sách các ID "không hợp lệ" dựa trên số lượng tương tác.
def get_illegal_ids_by_inter_num(df, field, max_num=None, min_num=None):
    if field is None:
        return set()
    if max_num is None and min_num is None:
        return set()

    max_num = max_num or np.inf
    min_num = min_num or -1

    ids = df[field].values
    inter_num = Counter(ids)
    ids = {
        id_ for id_ in inter_num if inter_num[id_] < min_num or inter_num[id_] > max_num
    }
    print(f"{len(ids)} illegal_ids_by_inter_num, field={field}")

    return ids


# Đây là hàm quan trọng nhất, thực hiện lọc dữ liệu theo vòng lặp cho đến khi mọi User và Item đều đạt chuẩn "5-core".
def filter_by_k_core(df):
    while True:
        ban_users = get_illegal_ids_by_inter_num(
            df, field=learner_id, max_num=None, min_num=min_u_num
        )
        ban_items = get_illegal_ids_by_inter_num(
            df, field=course_id, max_num=None, min_num=min_i_num
        )
        if len(ban_users) == 0 and len(ban_items) == 0:
            return

        dropped_inter = pd.Series(False, index=df.index)
        if learner_id:
            dropped_inter |= df[learner_id].isin(ban_users)
        if course_id:
            dropped_inter |= df[course_id].isin(ban_items)
        print(f"{len(dropped_inter)} dropped interactions")
        df.drop(df.index[dropped_inter], inplace=True)

In [12]:
k_core = 5
filter_by_k_core(df)
print(f"k-core shape: {df.shape}")
print(f"shape after k-core: {df.shape}")
df[:2]

3204490 illegal_ids_by_inter_num, field=userID
130284 illegal_ids_by_inter_num, field=itemID
5950878 dropped interactions
10362 illegal_ids_by_inter_num, field=userID
37994 illegal_ids_by_inter_num, field=itemID
1444267 dropped interactions
16939 illegal_ids_by_inter_num, field=userID
963 illegal_ids_by_inter_num, field=itemID
1323971 dropped interactions
750 illegal_ids_by_inter_num, field=userID
1543 illegal_ids_by_inter_num, field=itemID
1255986 dropped interactions
1322 illegal_ids_by_inter_num, field=userID
89 illegal_ids_by_inter_num, field=itemID
1247130 dropped interactions
66 illegal_ids_by_inter_num, field=userID
118 illegal_ids_by_inter_num, field=itemID
1241534 dropped interactions
119 illegal_ids_by_inter_num, field=userID
11 illegal_ids_by_inter_num, field=itemID
1240803 dropped interactions
8 illegal_ids_by_inter_num, field=userID
7 illegal_ids_by_inter_num, field=itemID
1240283 dropped interactions
1 illegal_ids_by_inter_num, field=userID
0 illegal_ids_by_inter_num, fie

,userID,itemID,rating,timestamp
30,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B086QM7FVT,3,1657839829629
31,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B017IQZ9OK,5,1655862402295


In [13]:
df["reviewerID"] = df["userID"].copy()
df["asin"] = df["itemID"].copy()

In [21]:
df

,userID,itemID,rating,timestamp,reviewerID,asin
30,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B086QM7FVT,3,1657839829629,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B086QM7FVT
31,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B017IQZ9OK,5,1655862402295,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B017IQZ9OK
32,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B08FZJ3YHH,4,1655860597696,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B08FZJ3YHH
33,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B082WJTFRR,5,1655860079499,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B082WJTFRR
34,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B004JU0H6O,4,1655859477958,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B004JU0H6O
...,...,...,...,...,...,...
6019558,AGMBTNN5HOKKKXINEATASO5OO35A,B08MWSBLCQ,5,1684845791986,AGMBTNN5HOKKKXINEATASO5OO35A,B08MWSBLCQ
6019560,AGMBTNN5HOKKKXINEATASO5OO35A,B086Z727MT,5,1659931693722,AGMBTNN5HOKKKXINEATASO5OO35A,B086Z727MT
6019561,AGMBTNN5HOKKKXINEATASO5OO35A,B01INO7AGG,4,1659931589085,AGMBTNN5HOKKKXINEATASO5OO35A,B01INO7AGG
6019562,AGMBTNN5HOKKKXINEATASO5OO35A,B0775VH1HH,4,1659931365259,AGMBTNN5HOKKKXINEATASO5OO35A,B0775VH1HH


## Re-index


In [14]:
df.reset_index(drop=True, inplace=True)

In [15]:
learner_id

'userID'

In [16]:
uid_field, iid_field = learner_id, course_id

uni_users = pd.unique(df[uid_field])
uni_items = pd.unique(df[iid_field])

u_id_map = {k: i for i, k in enumerate(uni_users)}
i_id_map = {k: i for i, k in enumerate(uni_items)}

df[uid_field] = df[uid_field].map(u_id_map)
df[iid_field] = df[iid_field].map(i_id_map)
df[uid_field] = df[uid_field].astype(int)
df[iid_field] = df[iid_field].astype(int)

In [17]:
df

,userID,itemID,rating,timestamp,reviewerID,asin
0,0,0,3,1657839829629,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B086QM7FVT
1,0,1,5,1655862402295,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B017IQZ9OK
2,0,2,4,1655860597696,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B08FZJ3YHH
3,0,3,5,1655860079499,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B082WJTFRR
4,0,4,4,1655859477958,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B004JU0H6O
...,...,...,...,...,...,...
1240214,150672,13548,5,1684845791986,AGMBTNN5HOKKKXINEATASO5OO35A,B08MWSBLCQ
1240215,150672,27903,5,1659931693722,AGMBTNN5HOKKKXINEATASO5OO35A,B086Z727MT
1240216,150672,13132,4,1659931589085,AGMBTNN5HOKKKXINEATASO5OO35A,B01INO7AGG
1240217,150672,29043,4,1659931365259,AGMBTNN5HOKKKXINEATASO5OO35A,B0775VH1HH


In [18]:
df.to_parquet(os.path.join(PATH, "step1_rating_to_inter", "df_inter.parquet"), index=False)

## Reload


In [19]:
indexed_df = pd.read_parquet(os.path.join(PATH, "step1_rating_to_inter", "df_inter.parquet"))
print(f"shape: {indexed_df.shape}")
indexed_df[:4]

shape: (1240219, 6)


,userID,itemID,rating,timestamp,reviewerID,asin
0,0,0,3,1657839829629,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B086QM7FVT
1,0,1,5,1655862402295,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B017IQZ9OK
2,0,2,4,1655860597696,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B08FZJ3YHH
3,0,3,5,1655860079499,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B082WJTFRR


In [20]:
u_uni = indexed_df[learner_id].unique()
c_uni = indexed_df[course_id].unique()

print(f"# of unique learners: {len(u_uni)}")
print(f"# of unique courses: {len(c_uni)}")

print("min/max of unique learners: {0}/{1}".format(min(u_uni), max(u_uni)))
print("min/max of unique courses: {0}/{1}".format(min(c_uni), max(c_uni)))

# of unique learners: 150673
# of unique courses: 35997
min/max of unique learners: 0/150672
min/max of unique courses: 0/35996
